In [1]:
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from dotenv import load_dotenv
import os
import warnings
import logging
from contextlib import redirect_stdout, redirect_stderr
from io import StringIO
from pathlib import Path

In [2]:
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

warnings.filterwarnings("ignore")

logging.getLogger().setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("torch").setLevel(logging.ERROR)

In [3]:
load_dotenv()

# Directory that contains your PDF files
PDF_DIRECTORY   = os.getenv("PDF_DIRECTORY", "PDF_Documents")
CHROMA_DIR      = os.getenv("CHROMA_DIR",    "chroma_db")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "credit_risk_corpus")

# Create the PDF directory if it doesn't exist yet
Path(PDF_DIRECTORY).mkdir(parents=True, exist_ok=True)
print(f"PDF source  : {os.path.abspath(PDF_DIRECTORY)}")
print(f"ChromaDB dir: {os.path.abspath(CHROMA_DIR)}")
print(f"Collection  : {COLLECTION_NAME}")


PDF source  : e:\Credit Risk Data Science\RAG Llama 3.2 1B\PDF_Documents
ChromaDB dir: e:\Credit Risk Data Science\RAG Llama 3.2 1B\chroma_db
Collection  : credit_risk_corpus


In [4]:
def load_pdf_documents(directory: str):
    """Load every PDF in *directory* and split into chunks."""
    pdf_files = list(Path(directory).rglob("*.pdf"))
    if not pdf_files:
        print(f"[WARNING] No PDF files found in '{directory}'.")
        return []

    print(f"Found {len(pdf_files)} PDF file(s):")
    for p in pdf_files:
        print(f"  - {p.name}")

    loader = PyPDFDirectoryLoader(directory)
    raw_docs = loader.load()
    print(f"\nLoaded {len(raw_docs)} raw page(s) from PDFs.")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        length_function=len,
        separators=["\n\n", "\n", ".", "!", "?", " ", ""]
    )
    chunks = splitter.split_documents(raw_docs)
    print(f"Split into {len(chunks)} chunk(s).")
    return chunks

In [5]:
# Preview first 3 chunks
chunks = load_pdf_documents(PDF_DIRECTORY)
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Source : {chunk.metadata.get('source', 'N/A')}  |  Page: {chunk.metadata.get('page', 'N/A')}")
    print(chunk.page_content[:300])

Found 9 PDF file(s):
  - Bakong policy.pdf
  - Basel II Framework.pdf
  - Basel III Finalising.pdf
  - Basel III Summary.pdf
  - Credit_risk_ML.pdf
  - IFRS9.pdf
  - national bank info.pdf
  - national bank policy.pdf
  - term of National bank Cambodia.pdf

Loaded 801 raw page(s) from PDFs.
Split into 2890 chunk(s).

--- Chunk 1 ---
Source : PDF_Documents\Bakong policy.pdf  |  Page: 0
Bak ong as a payment s witch
Mer chant - P r esented
L eng Ser e yw ath
A single QR C ode f or r eceiving payment fr om any mobile banking apps
No v ember 2020
National Bank o f C ambodia

--- Chunk 2 ---
Source : PDF_Documents\Bakong policy.pdf  |  Page: 1
o f C ontent
What is KHQR?
01 1
2
3
4
10
Bene f ts o f KHQR
02
Ho w it w ork s
03
Ho w to implement
04
F A Qs
05

--- Chunk 3 ---
Source : PDF_Documents\Bakong policy.pdf  |  Page: 2
What is KHQR?
The s tandar diz ation o f KHQR code speciﬁcation will help prOMOte wider use o f 
MOBILE r e tail P A YMENT S in C AMBODIA and pr o vide consis tent user e 

In [6]:
print(f"Total chunks to embed: {len(chunks)}")

Total chunks to embed: 2890


In [7]:
class E5Embeddings(HuggingFaceEmbeddings):
    def embed_documents(self, texts):
        texts = [f"passage: {t}" for t in texts]
        return super().embed_documents(texts)

    def embed_query(self, text):
        return super().embed_query(f"query: {text}")

In [ ]:
def build_chroma_from_pdf(pdf_dir: str, chroma_dir: str, collection: str):
    """Embed PDF chunks and persist them to ChromaDB."""
    docs = load_pdf_documents(pdf_dir)
    if not docs:
        print("No documents to embed. Please add PDF files and re-run.")
        return None

    print("\nInitialising embedding model (intfloat/multilingual-e5-base)…")
    with redirect_stdout(StringIO()), redirect_stderr(StringIO()):
        embedding = E5Embeddings(
            model_name="intfloat/multilingual-e5-base",
            model_kwargs={"device": "cpu"},   # change to "cuda" if a GPU is available
            encode_kwargs={"normalize_embeddings": True}
        )

    print(f"Embedding {len(docs)} chunk(s) and persisting to '{chroma_dir}'…")
    db = Chroma.from_documents(
        documents=docs,
        embedding=embedding,
        persist_directory=chroma_dir,
        collection_name=collection,
        collection_metadata={"hnsw:space": "cosine"}
    )

    persisted = db._collection.count()
    print(f"\nDone!  ChromaDB saved at '{os.path.abspath(chroma_dir)}'")
    print(f"Collection '{collection}' now contains {persisted} vector(s).")
    return db

# Run
db = build_chroma_from_pdf(PDF_DIRECTORY, CHROMA_DIR, COLLECTION_NAME)

Found 9 PDF file(s):
  - Bakong policy.pdf
  - Basel II Framework.pdf
  - Basel III Finalising.pdf
  - Basel III Summary.pdf
  - Credit_risk_ML.pdf
  - IFRS9.pdf
  - national bank info.pdf
  - national bank policy.pdf
  - term of National bank Cambodia.pdf

Loaded 801 raw page(s) from PDFs.
Split into 2890 chunk(s).

Initialising embedding model (intfloat/multilingual-e5-base)…
Embedding 2890 chunk(s) and persisting to 'chroma_db'…

Done!  ChromaDB saved at 'e:\Credit Risk Data Science\RAG Llama 3.2 1B\chroma_db'
Collection 'credit_risk_corpus' now contains 5605 vector(s).


In [ ]:
# Quick similarity test
if db is not None:
    test_query = "credit risk"
    results = db.similarity_search(test_query, k=3)
    print(f"Top {len(results)} result(s) for query: '{test_query}'\n")
    for i, doc in enumerate(results, 1):
        print(f"[{i}] Source: {doc.metadata.get('source', 'N/A')} | Page: {doc.metadata.get('page', 'N/A')}")
        print(f"     {doc.page_content[:200]}\n")

Top 3 result(s) for query: 'credit risk'

[1] Source: PDF_Documents\Credit_risk_ML.pdf | Page: 1
     Risks 2024, 12, 174 2 of 33
finalize the card issuance. A cutoff point will be established by creditors for credit scoring.
The institution might opt not to lend to the applicant if the score falls be

[2] Source: PDF_Documents\Credit_risk_ML.pdf | Page: 1
     Risks 2024, 12, 174 2 of 33
finalize the card issuance. A cutoff point will be established by creditors for credit scoring.
The institution might opt not to lend to the applicant if the score falls be

[3] Source: PDF_Documents\Credit_risk_ML.pdf | Page: 0
     Citation: Chang, Victor, Sharuga
Sivakulasingam, Hai Wang, Siu Tung
Wong, Meghana Ashok Ganatra, and
Jiabin Luo. 2024. Credit Risk
Prediction Using Machine Learning
and Deep Learning: A Study on
Credi



: 